# Setup the notebooke
## Define Minimum Maapping unit

In [1]:
#path setup
#since the notebook is in a subfolder, we need to add the src folder to the path
#The issue can be solved by installing the package in editable mode
from pathlib import Path
import sys
project_root = next(parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "src" / "ML_LC_Classifier").is_dir())
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
from ML_LC_Classifier import (
 apply_mmu
)
mmu_class = {
    1: 50,
    2: 50,
    3: 30,
    4: 30,
    5: 30,
    6: 10,
    7: 30,
    8: 30,
    9: 30,
    10: 30,
    11: 10,
    12: 10,
    13: 30,
    14: 30,
    15: 30,
    16: 30,
    17: 50,
}
#raster_path = r"C:\AFLICM_Test\Land Cover"


In [ ]:
from pathlib import Path

input_folder = Path(r"C:\AFLICM_Test\Land Cover\Landsat_new")
output_folder = input_folder / "MMU_processed_Landsat"
output_folder.mkdir(exist_ok=True)

raster_files = sorted(
    list(input_folder.glob("*.tif")) +
    list(input_folder.glob("*.tiff"))
)

outputs = list(map(
    lambda src: apply_mmu(
        src,
        output_folder / f"{src.stem}_MMU.tif",
        table=mmu_class,
        iterations=5,
    ),
    raster_files,
))

outputs

In [ ]:
from ML_LC_Classifier import apply_transition_consistency

# Define your own impossible transitions as (from_class, to_class)
impossible = [
    (2, 1),   # secondary → primary forest
    (7, 1),   # coffee → primary forest
    (16, 2),  # built-up → secondary forest
    (16, 12), # built-up → cropland
    (13, 1),  # grassland → primary forest
    (16, 14), # built-up → shrubland
    (1, 14), # Primary forest - shrubland

]

apply_transition_consistency(
    t1_path= r"C:\AFLICM_Test\Land Cover\Final Deliv\Post_Process_LULC_S2_2020_embedding.tif",
    t2_path= r"C:\AFLICM_Test\Land Cover\Final Deliv\LULC_S2_2025_embedding_unprocess_MMU.tif",
    output_path=r"C:\AFLICM_Test\Land Cover\Final Deliv\LULC_S2_2020_embedding_corrected.tif",
    forbidden=impossible,
    #flag_path="transition_flags.tif",       # optional
    #stable_mask_path="stable_areas.tif",    # optional
)


# Calculate area from land cover

In [2]:
from pathlib import Path
import matplotlib.pyplot as plt
from ML_LC_Classifier import calculate_class_area

#define list of land cover class and color
class_spec = {
    1:  {"name": "Primary Forest",        "color": "#00441B"},
    2:  {"name": "Secondary Forest",      "color": "#238B45"},
    3:  {"name": "Mangrove Forest",       "color": "#006D2C"},
    4:  {"name": "Coastal Forest",        "color": "#66C2A4"},
    5:  {"name": "Teak Plantation",       "color": "#8C6D31"},
    6:  {"name": "Eucalyptus Plantation", "color": "#A6761D"},
    7:  {"name": "Coffee Agroforestry",   "color": "#7FC97F"},
    8:  {"name": "Cocoa Agroforestry",    "color": "#66A061"},
    9:  {"name": "Coconut Agroforestry",  "color": "#41AB5D"},
    10: {"name": "Mixed Garden",          "color": "#A1D76A"},
    11: {"name": "Paddy Field",           "color": "#C7E9B4"},
    12: {"name": "Other Cropland",        "color": "#FDD835"},
    13: {"name": "Grassland",             "color": "#D9EF8B"},
    14: {"name": "Shrubland",             "color": "#C2A878"},
    15: {"name": "Cleared Land",          "color": "#d7d6d6"},
    16: {"name": "Built-up",              "color": "#DB310D"},
    17: {"name": "Water",                 "color": "#0e8acc"},
}
#folder
from pathlib import Path

lulc_folder = Path(r"C:\AFLICM_Test\Land Cover\Final Deliv")
output_folder = Path(r"C:\AFLICM_Test\Land Cover\Analysis")
output_folder.mkdir(parents=True, exist_ok=True)

In [3]:
all_results = {}
for tif in sorted(lulc_folder.glob("*.tif")):
    print(f"Processing {tif.name} ...")

    result = calculate_class_area(
        tif,
        class_spec=class_spec,
        csv_path=output_folder / f"{tif.stem}_area.csv",
        title=tif.stem,
    )

    all_results[tif.stem] = result
    result.figure.savefig(output_folder / f"{tif.stem}_area.png", dpi=150)
    plt.close(result.figure)   # avoid figure accumulation in long loops

    display(result.area_ha)    # show table inline in the notebook

Processing LULC_Landsat_2005_LT_CCDC_MMU.tif ...


,class_id,class_name,pixel_count,area_ha
0,1,Primary Forest,2980828,268274.52
1,2,Secondary Forest,4284309,385587.81
2,3,Mangrove Forest,43074,3876.66
3,4,Coastal Forest,291206,26208.54
4,5,Teak Plantation,4282,385.38
5,6,Eucalyptus Plantation,356645,32098.05
6,7,Coffee Agroforestry,483622,43525.98
7,8,Cocoa Agroforestry,9655,868.95
8,9,Coconut Agroforestry,172309,15507.81
9,10,Mixed Garden,1157565,104180.85


Processing LULC_Landsat_2015_LT_CCDC_MMU.tif ...


,class_id,class_name,pixel_count,area_ha
0,1,Primary Forest,2580492,232244.28
1,2,Secondary Forest,4156353,374071.77
2,3,Mangrove Forest,34710,3123.90
3,4,Coastal Forest,140420,12637.80
4,5,Teak Plantation,7050,634.50
5,6,Eucalyptus Plantation,445843,40125.87
6,7,Coffee Agroforestry,429069,38616.21
7,8,Cocoa Agroforestry,12462,1121.58
8,9,Coconut Agroforestry,164655,14818.95
9,10,Mixed Garden,980460,88241.40


Processing LULC_Landsat_2025_LT_CCDC_MMU.tif ...


...

,class_id,class_name,pixel_count,area_ha
0,1,Primary Forest,33480870,334808.70
1,2,Secondary Forest,36299333,362993.33
2,3,Mangrove Forest,1005449,10054.49
3,4,Coastal Forest,3287441,32874.41
4,5,Teak Plantation,1464513,14645.13
5,6,Eucalyptus Plantation,4802287,48022.87
6,7,Coffee Agroforestry,7035716,70357.16
7,8,Cocoa Agroforestry,351716,3517.16
8,9,Coconut Agroforestry,1438293,14382.93
9,10,Mixed Garden,3427236,34272.36


Processing Post_Process_LULC_S2_2020_MMU.tif ...


,class_id,class_name,pixel_count,area_ha
0,1,Primary Forest,40614846,406148.46
1,2,Secondary Forest,31421260,314212.60
2,3,Mangrove Forest,427367,4273.67
3,4,Coastal Forest,3288491,32884.91
4,5,Teak Plantation,4629336,46293.36
5,6,Eucalyptus Plantation,3693068,36930.68
6,7,Coffee Agroforestry,4865751,48657.51
7,8,Cocoa Agroforestry,704117,7041.17
8,9,Coconut Agroforestry,1803495,18034.95
9,10,Mixed Garden,2579673,25796.73


Processing Post_Process_LULC_S2_2025_MMU.tif ...


,class_id,class_name,pixel_count,area_ha
0,1,Primary Forest,40121377,401213.77
1,2,Secondary Forest,31954413,319544.13
2,3,Mangrove Forest,452850,4528.50
3,4,Coastal Forest,3344073,33440.73
4,5,Teak Plantation,5046393,50463.93
5,6,Eucalyptus Plantation,3680297,36802.97
6,7,Coffee Agroforestry,4952502,49525.02
7,8,Cocoa Agroforestry,895202,8952.02
8,9,Coconut Agroforestry,1988861,19888.61
9,10,Mixed Garden,2552498,25524.98
